# 50 · Three population models over the census districts, against rasterio

**Question.** Fieldwork experiment 209 totals WorldPop 2025, GRID3 2025 and GHS-POP 2025 over Rwanda's 30 and Botswana's
28 census districts with Sample a raster, and sets each district's share beside the 2022 census. Fieldwork's own test
compared those shares rounded to 0.1 %. Does an independent reading of the same files give the same cells, the same
totals and the same shares, unrounded?

**Independent.** rasterio reads each raster whole; GDAL's rasterizer (`geometry_mask`, a cell is in when its centre is)
marks each district's cells; NumPy sums them in float64. Nothing of Fieldwork's code is read.

**Criteria, fixed before running.** Each district's count of cells with a value equal; each total within 1e-9 relative;
each share within 1e-12.

**Where it runs.** On a computer holding the downloads (POP_FILES), each file checked by digest first. The rasters are
tens of megabytes each and are not redistributed here, so the browser kernel reports that it did not run.

In [ ]:
NEEDED = ["numpy", "rasterio"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the environment as installed.")

## Parity

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_50_population_districts import run

report = run(Path.cwd() / "fixtures" / "population-districts-001.json")
print("agrees:", report["agrees"], report.get("notRun", ""))
if report["agrees"] is not None:
    c = report["checked"]
    print(f"{c['districts']} districts, {c['totals']} totals, {c['shares']} shares")
    print(f"largest relative gap in a total: {report['largest']['relativeTotal']:.1e}; in a share: {report['largest']['share']:.1e}")
    print("differs:", report["differs"] or "nowhere")

## What this cannot establish

**That any source is right**: both sides total the same cells. **That a district's cells are its people**: a cell is
counted whole by where its centre falls. **That GHS-POP's window, cut from the global file, is the window Fieldwork reads
by range**: Fieldwork's browser test reads it by range and gives the same shares to 0.1 %.

In [ ]:
for item in report["notEstablished"]:
    print("-", item)
print()
print("fixture :", report["fixture"]["id"], report["fixture"]["sha256"][:16])
e = report.get("environment")
if e:
    print("rasterio:", e["rasterio"], "· GDAL", e["gdal"], "· NumPy", e["numpy"])
    print("runtime :", e["runtime"])